# DistilBERT Sentiment Classification of the IMDb Movie Reviews Dataset

This notebook demonstrates how to fine-tune DistilBERT language model (a distilled, lighter version of BERT) for binary sentiment classification using the IMDb movie reviews dataset.

The IMDb dataset contains 50,000 movie reviews labeled as either positive or negative sentiment.

Dataset split:
- Training set: 25,000 reviews
- Test set: 25,000 reviews

Target variable: Sentiment (positive / negative)

## Install Dependencies

In [1]:
# Install the required Hugging Face libraries
!pip install -q transformers datasets accelerate

## Import Modules

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix
# Hugging Face libraries
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

## Load the IMDb Dataset

In [3]:
# Load the IMDb dataset from the Hugging Face Hub
IMDb_dataset = load_dataset("imdb")

In [4]:
# Inspect the dataset structure
print(IMDb_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [5]:
# Inspect the dataset structure
print(IMDb_dataset["train"].features)

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


In [6]:
# Display the first 4 movie reviews
for i in range(4):
    print("Text:", IMDb_dataset["train"][i]["text"])
    print("Label:", IMDb_dataset["train"][i]["label"])

Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far betwee

## Tokenize the Dataset

In [7]:
# We will use a pretrained DistilBERT model
MODEL_CHECKPOINT = "distilbert-base-uncased"

# Load the DistilBERT tokenizer from Hugging Face
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

In [8]:
# Function to tokenize each movie review
def tokenize_function(examples):
    """Tokenize only the text and not the label; truncate long reviews to 256 tokens, pad short review to 256 tokens"""
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

# Apply tokenization to the entire dataset, use batched for faster tokenization
tokenized_datasets = IMDb_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [9]:
# Inspect the dataset after tokenization
print("Column names:", tokenized_datasets["train"].column_names)

Column names: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [10]:
# Inspect the first movie review
print(tokenized_datasets["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [11]:
# Text of the first review
print("Text:", tokenized_datasets["train"][0]["text"])
# Label of the first review
print("Label:", tokenized_datasets["train"][0]["label"])
# Tokens for each word or sub-word of the first review
print("Input IDs:", tokenized_datasets["train"][0]["input_ids"])
# Attention mask uses 1 and 1 to indicate if the text is padded
print("Attention Mask:", tokenized_datasets["train"][0]["attention_mask"])

Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far betwee

## Build the DistilBERT Classification Model

In [12]:
# Load the pre-trained DistilBERT model with a binary classification head
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Define Evaluation Metrics

In [13]:
# Function to compute accuracy during training
def compute_metrics(eval_pred):
    """Compute accuracy from model logits and ground-truth labels."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

## Configure Training Arguments

In [14]:
# Define training arguments for the model
training_args = TrainingArguments(
    # Directory to save model checkpoints and logs
    output_dir="./distilbert-imd",
    # Number of training epochs
    num_train_epochs=2,
    # Batch size for training
    per_device_train_batch_size=64,
    # Batch size for evaluation
    per_device_eval_batch_size=64,
    # Evaluation strategy - evaluate the model at the end of each epoch
    eval_strategy="epoch",
    # Optimizer to use (Adam with weight decay)
    optim="adamw_torch",
    # Learning rate for the optimizer
    learning_rate=2e-5,
    # Weight decay to apply for regularization
    weight_decay=0.01,
    # Disable reporting to external tools (e.g., WandB, TensorBoard)
    report_to="none"
)

## Train the Model

In [15]:
# Initialize the Trainer with the required parameters
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [16]:
# Fine-tune DistilBERT on the IMDb training set
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.229277,0.907800
2,0.285614,0.227181,0.909720


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Evaluate the Model

In [17]:
# Evaluate the model on the test dataset
test_accuracy = trainer.evaluate(tokenized_datasets["test"])

# Print the evaluation metrics
print(test_accuracy)

{'eval_loss': 0.22718141973018646, 'eval_accuracy': 0.90972, 'eval_runtime': 190.2066, 'eval_samples_per_second': 131.436, 'eval_steps_per_second': 2.056, 'epoch': 2.0}


## Predict on Custom Text

In [18]:
# # List of custom movies review to evaluate
custom_reviews = ["This movie was absolutely wonderful. The acting was superb and the plot kept me engaged throughout!",
    "A complete waste of time. The story made no sense and the characters were totally unconvincing.",
    "It was an okay film — nothing special, but not terrible either."]

In [19]:
# Tokenize the reviews, truncute/pad them to the same length for batch processing
inputs = tokenizer(custom_reviews, return_tensors="pt", padding=True, truncation=True).to(device=0)
inputs

{'input_ids': tensor([[  101,  2023,  3185,  2001,  7078,  6919,  1012,  1996,  3772,  2001,
         21688,  1998,  1996,  5436,  2921,  2033,  5117,  2802,   999,   102,
             0,     0,     0],
        [  101,  1037,  3143,  5949,  1997,  2051,  1012,  1996,  2466,  2081,
          2053,  3168,  1998,  1996,  3494,  2020,  6135,  4895,  8663,  6371,
          6129,  1012,   102],
        [  101,  2009,  2001,  2019,  3100,  2143,  1517,  2498,  2569,  1010,
          2021,  2025,  6659,  2593,  1012,   102,     0,     0,     0,     0,
             0,     0,     0]], device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1

In [20]:
# Forward pass through the model to make predictions
outputs = model(**inputs)

# Get predicted labels for each review
predicted_labels = outputs.logits.argmax(axis=-1).cpu().tolist()
print("Predicted Labels:", predicted_labels)

Predicted Labels: [1, 0, 1]
